# Geographic-Scale Training

Scaling sweep over the place-based top-N target pool: for each k in
`K_VALUES`, sample `N_SAMPLES` random k-subsets of the top-N species and
train one model per sample. The on-disk `non_target` class (assembled by
`dataset_build.ipynb`) already contains only non-top species + AudioSet
+ BirdNET no-bird, and `extras_into_other=0` ensures unchosen top-N
species are **never** folded into `non_target` for any run.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf

tf.get_logger().setLevel('ERROR')

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

In [ ]:
from building.scaling import (
    ScalingRunConfig,
    load_dataset_catalog,
    run_experiments,
    summarize_results,
    plot_summary,
)
from building.geographic_scale import scale_slug
from building.models import available_models, input_repr_for
print("Available models:", available_models())

# Parameters

In [ ]:
LAT, LON = 48.8566, 2.3522  # PARIS
RADIUS_KM = 50
N_TARGETS = 10

COLLECTION = scale_slug(LAT, LON, RADIUS_KM, n_targets=N_TARGETS)

N_SAMPLES = 10
K_VALUES = range(2, N_TARGETS + 1)
EPOCHS = 50
PATIENCE = 10
BATCH_SIZE = 32
SEED = 42
THRESHOLD = 0.5

BUILD_MODEL = "sincnet"

MODELS_DIR = ROOT / "models" / COLLECTION / BUILD_MODEL
RESULTS_FILE = MODELS_DIR / "results.jsonl"
print(f"collection: {COLLECTION}")
print(f"{BUILD_MODEL} input_repr={input_repr_for(BUILD_MODEL)}")
print(f"results: {RESULTS_FILE}")

In [ ]:
# extras_into_other=0 is the load-bearing override: it prevents the
# scaling runner from folding any unchosen top-N species into the "other"
# bucket. Combined with the dataset_build.ipynb invariant (top-N never
# pooled into non_target on disk), this guarantees top-N species are only
# ever used as targets.
config = ScalingRunConfig(
    collection=COLLECTION,
    build_model=BUILD_MODEL,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    threshold=THRESHOLD,
    models_dir=MODELS_DIR,
    results_file=RESULTS_FILE,
    extras_into_other=0,
)

catalog = load_dataset_catalog(collection=COLLECTION)

print(f"Loaded {len(catalog.class_names)} classes:")
print(catalog.class_names)

# Baselines (one model per top-N species, k=1)

In [ ]:
baseline_rows = run_experiments(
    catalog,
    config,
    run_baseline=True,
    run_scaling=False,
)
print(f"New baseline runs: {len(baseline_rows)}")
if baseline_rows:
    print("Last baseline row:")
    print(baseline_rows[-1])

In [ ]:
from building.scaling import print_baselines
print_baselines(catalog, RESULTS_FILE)

# Scaling sweep

In [ ]:
scaling_rows = run_experiments(
    catalog,
    config=config,
    n_samples=N_SAMPLES,
    k_values=K_VALUES,
    run_baseline=False,
    run_scaling=True,
)
print(f"New scaling runs: {len(scaling_rows)}")
if scaling_rows:
    print("Last scaling row:")
    print(scaling_rows[-1])

# Summary + plot

In [ ]:
baseline_metrics, summary_df = summarize_results(RESULTS_FILE)
print(f"Baseline recall: {baseline_metrics.recall:.4f}")
print(f"Baseline precision: {baseline_metrics.precision:.4f}")
summary_df

In [ ]:
plot_summary(summary_df, baseline=baseline_metrics)